# 🏏 IPL Match Predictor - Master Hackathon Notebook (v2.0)
## Optimized Ensemble Model with Pre-Submission Validation

This notebook focuses on **accuracy and validation**. It will:
1. Auto-discover your datasets.
2. Train and show you the **Model Score**.
3. Display the **full submission table** for your review before final output.

In [ ]:
# [1] Setup
!pip install xgboost scikit-learn pandas numpy requests -q
import pandas as pd
import numpy as np
import os, glob, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from IPython.display import display

warnings.filterwarnings('ignore')

## 🔍 Flexible Data Discovery
Searching for datasets in `/kaggle/input`...

In [ ]:
def find_file(filename):
    """Recursively find a file in Kaggle or Local folders"""
    search_paths = ['/kaggle/input/**', './**/*.csv', 'backend/data/*.csv']
    for pattern in search_paths:
        files = glob.glob(pattern, recursive=True)
        for f in files:
            if filename.lower() in f.lower():
                return f
    return None

FILES = {
    'train': find_file('train_IPL.csv') or find_file('match_summary.csv'),
    'lb': find_file('public_lb_matches.csv'),
    'sample': find_file('sample_submission.csv')
}

datasets = {}
for name, fpath in FILES.items():
    if fpath:
        datasets[name] = pd.read_csv(fpath)
        print(f"✅ Found {name}: {fpath} ({datasets[name].shape})")
    else:
        print(f"❌ Could not find {name} file!")

## 🧠 Model Training & Performance Score

In [ ]:
class MasterPredictor:
    def __init__(self):
        self.team_le = LabelEncoder()
        self.venue_le = LabelEncoder()
        self.scaler = StandardScaler()
        self.model = None
        self.classes = []

    def train_and_score(self, df):
        # Features
        all_teams = pd.concat([df['team_a'], df['team_b']]).unique()
        self.team_le.fit(all_teams)
        self.venue_le.fit(df['venue'].unique())
        
        X = np.array([[self.team_le.transform([r['team_a']])[0], 
                       self.team_le.transform([r['team_b']])[0], 
                       self.venue_le.transform([r['venue']])[0]] for _, r in df.iterrows()])
        
        le = LabelEncoder()
        y = le.fit_transform(df['outcome'])
        self.classes = le.classes_
        
        # Train-Test Split for Scoring
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        
        self.scaler.fit(X_train)
        X_tr_s = self.scaler.transform(X_train)
        X_val_s = self.scaler.transform(X_val)
        
        # Ensemble
        xgb = CalibratedClassifierCV(XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42), cv=3)
        xgb.fit(X_tr_s, y_train)
        
        # Scoring
        preds = xgb.predict(X_val_s)
        probs = xgb.predict_proba(X_val_s)
        
        print("="*40)
        print(f"📈 MODEL VALIDATION SCORE")
        print(f"Accuracy: {accuracy_score(y_val, preds)*100:.2f}%")
        print(f"Log Loss: {log_loss(y_val, probs):.4f}")
        print("="*40)
        
        self.model = xgb
        return self

    def predict_row(self, row):
        try:
            feat = np.array([[self.team_le.transform([row['team_a']])[0], 
                              self.team_le.transform([row['team_b']])[0], 
                              self.venue_le.transform([row['venue']])[0]]])
            feat_s = self.scaler.transform(feat)
            return self.model.predict_proba(feat_s)[0]
        except:
            return np.array([0.25, 0.25, 0.25, 0.25])

if 'train' in datasets:
    predictor = MasterPredictor().train_and_score(datasets['train'])

## 📊 Leaderboard Predictions (Review Before Submission)

In [ ]:
if 'lb' in datasets:
    print("Generating Leaderboard Predictions...")
    results = []
    for _, row in datasets['lb'].iterrows():
        probs = predictor.predict_row(row)
        results.append({
            'match_id': row.get('match_id', f"{row['team_a'][:3]}_{row['team_b'][:3]}"),
            'A_big': round(probs[0], 4), 'A_small': round(probs[1], 4),
            'B_big': round(probs[2], 4), 'B_small': round(probs[3], 4)
        })
    
    sub_df = pd.DataFrame(results)
    
    print("\n👀 PREVIEW OF YOUR SUBMISSION:")
    display(sub_df)
    
    sub_df.to_csv('submission.csv', index=False)
    print("\n✅ submission.csv saved and ready for download!")
else:
    print("⚠️ lb file missing - cannot show preview.")